# RAG Week!:

Working on an Expert Knowledge worker. An Expert Assistant that answers users question.

> Context: <br>
* To be used employees of an insurance tech company.
* AI assistant should be accurate and provide solutions with low cost.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path
import gradio as gr
import glob
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override = True)
openai_key = os.getenv("OPENAI_API_KEY")

if openai_key:
    print("The key has been loaded successfully!")
else:
    print("Key not found!")

Model = "gpt-5-nano"
openai = OpenAI()

## Reading the Employee data:

Created a dictionary where Key => "Employee name" and Value => "Employee details"

In [ ]:
Knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.split(" ")[-1]
    with open(filename, "r", encoding = "utf-8") as f:
        Knowledge[name.lower()] = f.read()
    

In [ ]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    name = Path(filename).stem.split(" ")[-1]
    with open(filename, "r", encoding = "utf-8") as f:
        Knowledge[name.lower()] = f.read()
    

In [ ]:
display(Markdown(Knowledge["healthllm"]))

### System Prompt:

In [ ]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

### Gathering relevant context from the Knowledge base:

In [ ]:
"""This function strips any special characters, symbols from the message first, then converts it into list of words from the original message. Then we check whether any of the words has any 
context stored in the 'Knowledge' dictionary. If there is, it is append to the 'relevant_info' List."""
def get_relevant_info(message):
    text = " ".join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    relevant_info = []
    for word in words:
        if word in Knowledge:
            relevant_info.append(Knowledge[word])
    return relevant_info

In [ ]:
# More Pythonic way of writing the function using list comprehension:

def get_relevant_info2(message):
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [Knowledge[word] for word in words if word in Knowledge]

In [ ]:
get_relevant_info2("Who is chen")

In [ ]:
get_relevant_info2("Who is chen and what is healthllm?")

## Creating an official function to wrap the entire process:

In [ ]:
def additional_context(message):
    relevant_context = get_relevant_info2(message)
    if not relevant_context:
        result = "There is no relevant information based on the given message in the knowledge base!"
    else:
        result = "Found relevant information based on user prompt/message."
        result+= "\n\n".join(relevant_context)
    return result

In [ ]:
print(additional_context("Who is chen and what is healthllm?"))

In [ ]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role":"system","content":system_message}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(
        model = Model,
        messages = messages
    )
    return response.choices[0].message.content

In [ ]:
view = gr.ChatInterface(chat, type = "messages").launch(inbrowser=True)